<a href="https://colab.research.google.com/github/juanjosecas/sleap-postprocessig/blob/main/SLEAP_Training_Inference_en_Google.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento e inferencia con datos de Google Drive



## Install SLEAP


In [ ]:
!python --version

!wget "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

!bash Miniforge3-$(uname)-$(uname -m).sh -b

!clear

!/root/miniforge3/bin/mamba create -y -n sleap -c conda-forge -c nvidia -c sleap -c anaconda sleap=1.3.3

Python 3.10.12
--2024-10-01 13:01:02--  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/conda-forge/miniforge/releases/download/24.7.1-2/Miniforge3-Linux-x86_64.sh [following]
--2024-10-01 13:01:03--  https://github.com/conda-forge/miniforge/releases/download/24.7.1-2/Miniforge3-Linux-x86_64.sh
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://objects.githubusercontent.com/github-production-release-asset-2e65be/221584272/182642f9-81e6-4ecb-aaaf-372358c0d883?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=releaseassetproduction%2F20241001%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20241001T130008Z&X-Amz-Expires=300&X-Amz-Signature=2dec767129d9c1a846186e32e5b51ca73d26d60fc212c3f9d43

In [ ]:
#!ls /root/miniforge3/envs/sleap/bin/

python_path = "/root/miniforge3/envs/sleap/bin/python"
sleap_diagnostic = "/root/miniforge3/envs/sleap/bin/sleap-diagnostic"

!{python_path} --version

!{sleap_diagnostic}

Python 3.7.12
2024-10-01 13:48:11.436834: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2024-10-01 13:48:11.436869: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-10-01 13:48:19.729516: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-10-01 13:48:19.730042: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib64-nvidia
2024-10-01 13:48:19.730288: W tensorflow/stream_executor/platform/default/ds

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib

# Chequear el funcionamiento de la GPU
print('==== Info de la GPU ====')
!nvidia-smi
print('\n')
print('==== Info del compilador CUDA ====')
!nvcc --version
print('\n')
print('==== Ubicación del compilador ====')
!type nvcc
print('\n')

import tensorflow as tf

# List all physical devices available to TensorFlow
physical_devices = tf.config.list_physical_devices()
print("All physical devices:", physical_devices)

# List all GPUs available to TensorFlow
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
else:
    print("No GPUs found")
    print('\n')



print('Iniciar la instalación de SLEAP')
!pip uninstall -qqq -y opencv-python opencv-contrib-python
!pip install -qqq "sleap[pypi]>=1.3.3"
print('Fin de la instalación de SLEAP\n\n')
print('\n')

try:
    importlib.import_module('sleap')
    print('Chequear versión instalada de SLEAP')

    import sleap

    sleap.versions()
    sleap.system_summary()
    print('\n')

except ImportError:
    print("SLEAP is not installed.")
    print('\n')

==== Info de la GPU ====
Tue Oct  1 11:56:28 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   61C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+

In [ ]:
mount_point = '/content/drive/'

print('Montar Google Drive')
print('Montar su Google Drive le permitirá acceder al paquete de trabajo de training cargado en este cuaderno. \nCuando se le solicite iniciar sesión en su cuenta de Google, otorgue acceso a Colab y copie el código de autorización en un campo a continuación (+ presione Enter.')
from google.colab import drive
drive.mount(mount_point)
print(f'Concluido Montar Google Drive en {mount_point}')

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.device(0))
print(torch.cuda.get_device_name(0))

True
1
0
Tesla T4


### Linkear la carpeta que contiene los archivos y labels

Tenemos que ir a nuestro Google Drive y encontrar la carpeta que contiene todos los archivos del proyecto. Cuando tenemos que armar la dirección o path de la carpeta, nuestro Drive siempre estará en la carpeta raíz '/content/drive/MyDrive/' a menos que hayamos configurado.

Luego, tenemos que conocer el nombre del archivo exportado desde SLEAP.

In [ ]:
# DONDE están los archivos del proyecto?

project_folder = "/content/drive/MyDrive/SLEAP"

import os

os.chdir(project_folder)
print('El contenido de la carpeta donde está el proyecto es:\n')
!ls -lh

!unzip "/content/drive/MyDrive/SLEAP/labels.v001.slp.training_job.zip"

!ls -lh




El contenido de la carpeta donde está el proyecto es:

total 1.8M
-rw------- 1 root root 1.8M Jun  4 22:19 labels.v001.slp.training_job.zip
Archive:  /content/drive/MyDrive/SLEAP/labels.v001.slp.training_job.zip
  inflating: train-script.sh         
  inflating: inference-script.sh     
  inflating: jobs.yaml               
  inflating: centered_instance.json  
  inflating: labels.v001.pkg.slp     
  inflating: centroid.json           
total 3.7M
-rw------- 1 root root 5.1K Jun  4 19:19 centered_instance.json
-rw------- 1 root root 5.0K Jun  4 19:19 centroid.json
-rw------- 1 root root 7.1K Jun  4 19:19 inference-script.sh
-rw------- 1 root root 5.5K Jun  4 19:19 jobs.yaml
-rw------- 1 root root 2.0M Jun  4 19:19 labels.v001.pkg.slp
-rw------- 1 root root 1.8M Jun  4 22:19 labels.v001.slp.training_job.zip
-rw------- 1 root root  113 Jun  4 19:19 train-script.sh


In [ ]:
training_file = "labels.v001.pkg.slp"

# Load the labels from a file
labels = sleap.Labels.load_file(training_file)

# Get the data and inspect the shape
print('Contenido del archivo de configuración del proyecto:')
print(labels)

Contenido del archivo de configuración del proyecto:
Labels(labeled_frames=3, videos=24, skeletons=1, tracks=0)


## Entrenamiento

Dependiendo del flujo de trabajo que hayas elegido en el cuadro de diálogo de entrenamiento, los nombres de archivo de configuración serán los siguientes:

- Para un enfoque de flujo de trabajo de abajo hacia arriba: multi_instance.json.
- Para un flujo de trabajo de arriba hacia abajo, tendrás un perfil diferente para cada uno de los modelos: centroid.json y centered_instance.json.
- Para un flujo de trabajo de un solo animal: single_instance.json.

### Nota sobre el proceso de entrenamiento

Cuando comiences el entrenamiento, primero verás los parámetros de entrenamiento y luego la pérdida de entrenamiento y validación para cada época de entrenamiento. Tan pronto como estés satisfecho con la pérdida de validación que ves para una época durante el entrenamiento, puedes detener el entrenamiento haciendo clic en el botón de detención. La versión del modelo con la pérdida de validación más baja se guarda durante el entrenamiento y es la que se utilizará para la inferencia. Si no detienes el entrenamiento, se ejecutará durante 200 épocas o hasta que la pérdida de validación deje de mejorar durante cierto número de épocas (controlado por los campos de early_stopping en el perfil de entrenamiento). Si en lugar de elegir el enfoque de abajo hacia arriba has optado por el flujo de trabajo de arriba hacia abajo (con dos configuraciones de entrenamiento), deberás invocar dos trabajos de entrenamiento separados en secuencia:

```
!sleap-train centroid.json colab.pkg.slp

!sleap-train centered_instance.json colab.pkg.slp
```





```
usage: sleap-train [-h] [--video-paths VIDEO_PATHS] [--val_labels VAL_LABELS]
                   [--test_labels TEST_LABELS] [--tensorboard] [--save_viz]
                   [--zmq] [--run_name RUN_NAME] [--prefix PREFIX]
                   [--suffix SUFFIX]
                   training_job_path [labels_path]

positional arguments:
  training_job_path     Path to training job profile JSON file.
  labels_path           Path to labels file to use for training. If specified,
                        overrides the path specified in the training job
                        config.

optional arguments:
  -h, --help            show this help message and exit
  --video-paths VIDEO_PATHS
                        List of paths for finding videos in case paths inside
                        labels file are not accessible.
  --val_labels VAL_LABELS, --val VAL_LABELS
                        Path to labels file to use for validation. If
                        specified, overrides the path specified in the
                        training job config.
  --test_labels TEST_LABELS, --test TEST_LABELS
                        Path to labels file to use for test. If specified,
                        overrides the path specified in the training job
                        config.
  --base_checkpoint BASE_CHECKPOINT
                        Path to base checkpoint (directory containing best_model.h5)
                        to resume training from.
  --tensorboard         Enable TensorBoard logging to the run path if not
                        already specified in the training job config.
  --save_viz            Enable saving of prediction visualizations to the run
                        folder if not already specified in the training job
                        config.
  --zmq                 Enable ZMQ logging (for GUI) if not already specified
                        in the training job config.
  --run_name RUN_NAME   Run name to use when saving file, overrides other run
                        name settings.
  --prefix PREFIX       Prefix to prepend to run name.
  --suffix SUFFIX       Suffix to append to run name.
  --cpu                 Run training only on CPU. If not specified, will use
                        available GPU.
  --first-gpu           Run training on the first GPU, if available.
  --last-gpu            Run training on the last GPU, if available.
  --gpu GPU             Run training on the i-th GPU on the system. If 'auto', run on
                        the GPU with the highest percentage of available memory.
```



## Un solo animal

In [ ]:
# ====== PARA SINGLEANIMAL =======
print('==== Inicio de entrenamiento SINGLE ANIMAL  ====')
!sleap-train --gpu 0 single_instance.json {training_file}
print('==== Fin de entrenamiento SINGLE ANIMAL ====')


==== Inicio de entrenamiento SINGLE ANIMAL  ====
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
2024-03-15 16:25:21.126676: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/lib64-nvidia
INFO:sleap.nn.training:Versions:
SLEAP: 1.3.2
TensorFlow: 2.8.4
Numpy: 1.22.4
Python: 3.10.12
OS: Linux-6.1.58+-x86_64-with-glibc2.35
INFO:sleap.nn.training:Training labels file: labels.v001.pkg.slp
INFO:sleap.nn.training:Training profile: single_instance.json
INFO:sleap.nn.training:
INFO:sleap.nn.training:Arguments:
INFO:sleap.nn.training:{
    "training_job_path": "single_instance.json",
    "labels_path": "labels.v001.pkg.slp",
    "video_paths": [
        ""
    ],
    "val_labels": null,
    "test_labels": null

## Multi animal

In [ ]:
# ====== PARA MULTIANIMAL =========

print('####### Entrenamiento del Centroide ########\n\n')
!sleap-train centroid.json {training_file}
print('')
print('####### Centered instance ########\n\n')
!sleap-train centered_instance.json {training_file}
print('')
print('==== Fin de entrenamiento MULTIANIMAL ====')


####### Entrenamiento del Centroide ########


INFO:numexpr.utils:NumExpr defaulting to 2 threads.
2024-06-04 22:53:16.692377: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/lib64-nvidia
INFO:sleap.nn.training:Versions:
SLEAP: 1.3.0
TensorFlow: 2.8.4
Numpy: 1.22.4
Python: 3.10.12
OS: Linux-6.1.85+-x86_64-with-glibc2.35
INFO:sleap.nn.training:Training labels file: labels.v001.pkg.slp
INFO:sleap.nn.training:Training profile: centroid.json
INFO:sleap.nn.training:
INFO:sleap.nn.training:Arguments:
INFO:sleap.nn.training:{
    "training_job_path": "centroid.json",
    "labels_path": "labels.v001.pkg.slp",
    "video_paths": [
        ""
    ],
    "val_labels": null,
    "test_labels": null,
    "base_chec

In [ ]:
# DEFINAMOS dónde están los modelos a usar

centroid_model = "/content/drive/MyDrive/Zebrafish/models/simple230613_184026.centroid"
centered_instance_model = "/content/drive/MyDrive/Zebrafish/models/simple230613_184026.centered_instance"

single_model = "/content/drive/MyDrive/lorena\ rela/SLEAP/Version1-30-06-2023/models/rela230630_110127.single_instance"

In [ ]:
from moviepy.editor import *

def convert_video(video_file, output, new_fps=24, start=0, end=30, fraction=1.0):
    # cargar el video  como clip de Moviepy
    clip = VideoFileClip(video_file)
    # getting only first final - comienzo seconds
    clip_sub = clip.subclip(start, end)
    clip_resized = clip_sub.resize(fraction)
    # saving the clip
    clip_resized.write_videofile(output, fps=new_fps)
    print('Finalizada la conversión de',video_file,'a',output)

def track_video(video_file, output_detection_file, instance_count=3, track_window=13, batch_size=20, peak_threshold=0.35, min_match_points=1, kalman=21):
    !sleap-track {video_file} \
        --tracking.tracker simple \
        -m {centroid_model} \
        -m {centered_instance_model} \
        --verbosity rich \
        --tracking.similarity 'instance' \
        --batch_size {batch_size} \
        --tracking.target_instance_count {instance_count} \
        --tracking.clean_instance_count {instance_count} \
        --tracking.track_window {track_window} \
        --tracking.min_match_points {min_match_points} \
        --tracking.kf_init_frame_count {kalman} \
        -o {output_detection_file}

def track_single(video_file, output_detection_file, instance_count=1, tracker='simple' ,track_window=13, batch_size=20, peak_threshold=0.35, min_match_points=1):
    !sleap-track {video_file} \
        --tracking.tracker {tracker} \
        -m {single_model} \
        --verbosity rich \
        --tracking.similarity instance \
        --batch_size {batch_size} \
        --tracking.target_instance_count {instance_count} \
        --tracking.track_window {track_window} \
        --tracking.min_match_points {min_match_points} \
        -o {output_detection_file}


def convert(output_detection_file, output_json, output_h5, output_analysis):
    !sleap-convert {output_detection_file} --format json -o {output_json}
    !sleap-convert {output_detection_file} --format h5 -o {output_h5}
    !sleap-convert {output_detection_file} --format analysis -o {output_analysis}

def render(output_h5, labeled_video, marker_size=2, palette='five+', distinctly_color='nodes'):
    !sleap-render {output_h5} \
        --marker_size {marker_size} \
        --show_edges 1 \
        -o {labeled_video} \
        --palette {palette} \
        --distinctly_color {distinctly_color}

def movie_prop(video_file):
    clip = VideoFileClip(video_file)
    (w, h) = clip.size
    duration = clip.duration
    return w, h, duration



## Inferencia en videos

In [ ]:
"""
convert_video(
    video_file='videos/video1.mp4',
    output='video1_edit.mp4',
    new_fps=24,
    start=0,
    end=90,
    fraction=0.65
)

track_single(video_file='/content/drive/MyDrive/lorena\ rela/SLEAP/Version1-30-06-2023/videos/2020-02-26\ C-3.mp4',
            output_detection_file='2020-02-26_C-3.track.slp',
            instance_count=1,
            track_window=20,
            batch_size=10,
            peak_threshold=0.25,
            min_match_points=1
            )


convert(output_detection_file='2020-02-26_C-3.track.slp',
        output_json='2020-02-26_C-3.track.json',
        output_h5='2020-02-26_C-3.track.h5',
        output_analysis='2020-02-26_C-3.analysis.h5'
        )
"""
render(output_h5='2020-02-26_C-3.track.h5',
       marker_size=2,
       labeled_video='2020-02-26_C-3.labeled2.mp4',
       palette='five+',
       distinctly_color='nodes'
       )


INFO:numexpr.utils:NumExpr defaulting to 2 threads.
Saving config: /root/.sleap/1.3.1/preferences.yaml
Writing video with 40509 frame images...
INFO:sleap.io.visuals:Chunks: 633, chunk size: 64
Finished 64 frames in 3.0 s, fps = 21, approx 1897.8 s remaining
Finished 128 frames in 3.5 s, fps = 36, approx 1109.0 s remaining
Finished 192 frames in 4.0 s, fps = 48, approx 835.8 s remaining
Finished 256 frames in 4.5 s, fps = 56, approx 713.6 s remaining
Finished 320 frames in 5.0 s, fps = 64, approx 627.4 s remaining
Finished 384 frames in 5.4 s, fps = 71, approx 568.5 s remaining
Finished 448 frames in 5.9 s, fps = 76, approx 527.4 s remaining
Finished 512 frames in 6.4 s, fps = 80, approx 501.9 s remaining
Finished 576 frames in 6.9 s, fps = 83, approx 479.9 s remaining
Finished 640 frames in 7.4 s, fps = 86, approx 462.2 s remaining
Finished 704 frames in 8.1 s, fps = 87, approx 455.9 s remaining
Finished 768 frames in 8.5 s, fps = 90, approx 442.1 s remaining
Finished 832 frames in 9.